# ALPR — Plate Detector Training (Kaggle)

This notebook trains the YOLO plate detector on a Kaggle GPU instance.
It is the **only** place where model weights are produced; everything else
in the repository is about using those weights.

**Before running:**
1. Attach your dataset under `Dataset > Add data`. It will appear at
   `/kaggle/input/<dataset-slug>/`.
2. Enable GPU: `Settings > Accelerator > GPU T4 x2` (or P100).
3. Set `INPUT_ROOT` in cell 2 to your dataset slug.

**After running:**
Download `best.pt` from the output and copy it into your local `models/`
directory, then update `configs/default.yaml`:
```yaml
plate_detector:
  weights: plate_yolo26s.pt
```

In [ ]:
# Cell 1 — Install the ALPR package from the attached dataset or GitHub.
# On Kaggle, the simplest approach is to add this repository as a dataset
# (zip the repo, upload it) and install it from there.
#
# Alternative: clone from GitHub if the notebook has internet access.
import subprocess, sys

# Try local install first (repo attached as dataset).
REPO_PATH = "/kaggle/input/alpr-repo/ALPR_Upgraded"  # adjust to your slug
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", REPO_PATH, "-q"])
    print(f"Installed alpr from {REPO_PATH}")
except Exception:
    print("Local install failed — trying pip install from current dir")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "-q"])

In [ ]:
# Cell 2 — Configuration. Edit these before running.
from pathlib import Path

INPUT_ROOT   = Path("/kaggle/input/vietnamese-license-plate")  # your dataset slug
WORKING_DIR  = Path("/kaggle/working")
BASE_WEIGHTS = "yolo26s.pt"   # starting checkpoint; smaller = faster
EPOCHS       = 100
IMGSZ        = 640
BATCH        = 16             # reduce to 8 if OOM
DEVICE       = "0"            # first GPU; "0,1" for two GPUs

print(f"Input : {INPUT_ROOT}")
print(f"Output: {WORKING_DIR}")

In [ ]:
# Cell 3 — Prepare the dataset.
#
# kaggle_setup() audits the data, splits it, and materialises it in copy mode
# so Ultralytics can write labels.cache freely.
from alpr.training import kaggle_setup

dataset_yaml = kaggle_setup(
    input_root=INPUT_ROOT,
    working_dir=WORKING_DIR,
    seed=42,
    ratios=(0.80, 0.10, 0.10),
)
print(f"Dataset YAML: {dataset_yaml}")

In [ ]:
# Cell 4 — Train.
#
# This is the expensive cell. On a T4 with 640px and batch=16:
#   100 epochs ≈ 2–3 hours for a 24k-image dataset.
# Early stopping (patience=20) exits sooner if val loss plateaus.
from alpr.training import TrainingConfig, train

config = TrainingConfig(
    dataset_yaml=dataset_yaml,
    base_weights=BASE_WEIGHTS,
    epochs=EPOCHS,
    patience=20,
    batch=BATCH,
    imgsz=IMGSZ,
    workers=2,              # Kaggle provides 2 CPUs
    device=DEVICE,
    half=True,              # FP16 is safe on T4/P100
    cache=False,            # Kaggle RAM is shared; don't hoard it
    project=str(WORKING_DIR / "runs"),
    name="plate_detector",
    exist_ok=True,
    seed=42,
    verbose=False,
)

best_weights = train(config)
print(f"\nBest weights saved to: {best_weights}")

In [ ]:
# Cell 5 — Quick validation on the held-out test split.
#
# This is a sanity check only: the real evaluation (mAP, recall curves)
# should be done locally with scripts/evaluate_ocr.py after downloading
# the weights and running inference on the test set.
from ultralytics import YOLO
import yaml

model = YOLO(str(best_weights))
metrics = model.val(
    data=str(dataset_yaml),
    split="test",
    imgsz=IMGSZ,
    device=DEVICE,
    verbose=False,
)

print(f"mAP50       : {metrics.box.map50:.4f}")
print(f"mAP50-95    : {metrics.box.map:.4f}")
print(f"Precision   : {metrics.box.mp:.4f}")
print(f"Recall      : {metrics.box.mr:.4f}")

In [ ]:
# Cell 6 — Copy best.pt to the output directory root.
#
# Kaggle saves everything under /kaggle/working. The output panel shows
# files in the root; nested files need a manual download.
import shutil
dest = WORKING_DIR / "plate_yolo26s.pt"
shutil.copy2(best_weights, dest)
print(f"Ready to download: {dest}")
print()
print("Next steps:")
print("  1. Download plate_yolo26s.pt from the Output panel.")
print("  2. Copy to: models/plate_yolo26s.pt")
print("  3. Update configs/default.yaml: plate_detector.weights: plate_yolo26s.pt")